# Hindi-to-Spanish Speech-to-Speech Translation (S2ST)
### PyTorch Custom Transformer Implementation with Pretrained Audio Components

This notebook allows you to develop, train, evaluate, and test the Hindi-to-Spanish S2ST system on Google Colab GPUs (T4, V100, A100).

## 1. Environment & GPU Acceleration Verification

In [ ]:
# Check CUDA GPU availability and specifications
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"Supports FP16 AMP: True")
    print(f"Supports BF16 AMP: {torch.cuda.is_bf16_supported()}")
else:
    print("Warning: Running on CPU. For GPU acceleration, switch runtime: Runtime -> Change runtime type -> T4 GPU.")

## 2. Mount Google Drive (Optional, for persistent checkpoints)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except Exception as e:
    print("Not running in Google Colab or Drive mount skipped:", e)

## 3. Install Dependencies

In [ ]:
# Install project dependencies
!pip install -q pyyaml soundfile librosa transformers sentencepiece datasets sacrebleu pytest tqdm

## 4. Run Architecture Unit Tests
Verify that the custom Transformer attention, positional encodings, encoder/decoder stacks, and causal masking pass all mathematical assertions.

In [ ]:
!pytest tests/ -v

## 5. Inspect Model Architecture and Parameter Count

In [ ]:
import yaml
from src.models.transformer import Seq2SeqTransformer
from src.utils.device import get_device

with open("configs/default_config.yaml", "r") as f:
    config = yaml.safe_load(f)

device = get_device(config["system"]["device"])
m_cfg = config["model"]
a_cfg = config["audio"]

model = Seq2SeqTransformer(
    src_vocab_size=config["text"]["src_vocab_size"],
    tgt_vocab_size=config["text"]["tgt_vocab_size"],
    d_model=m_cfg["d_model"],
    num_heads=m_cfg["nhead"],
    num_encoder_layers=m_cfg["num_encoder_layers"],
    num_decoder_layers=m_cfg["num_decoder_layers"],
    dim_feedforward=m_cfg["dim_feedforward"],
    dropout=m_cfg["dropout"],
    activation=m_cfg["activation"],
    audio_input_dim=a_cfg["n_mels"],
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Device: {device}")
print(f"Total Model Parameters: {total_params:,}")
print(f"Trainable Parameters:  {trainable_params:,}")

## 6. End-to-End Inference Demo
Translates an input audio segment (or synthetic tone/sample) and synthesizes Spanish audio using Meta MMS-TTS.

In [ ]:
import IPython.display as ipd
from src.audio.vocoder import SpanishTTSPipeline

# Test pretrained Spanish TTS component
tts = SpanishTTSPipeline(model_id="facebook/mms-tts-spa")
sample_spanish_text = "Hola, esta es una traducción de prueba del hindi al español."
waveform = tts.synthesize(sample_spanish_text)

print(f"Synthesized Spanish speech: '{sample_spanish_text}'")
ipd.Audio(waveform.numpy(), rate=16000)